In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

import csv
import copy

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split

import torchvision.models as models

from tqdm.auto import tqdm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
save_dir = "/content/drive/MyDrive/KidneyTumorClassification/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

log_file = os.path.join(save_dir, "training_log_resnet.csv")
best_model_path = os.path.join(save_dir, "best_model_resnet.pth")
last_model_path = os.path.join(save_dir, "last_model_resnet.pth")

In [ ]:
!unzip -q /content/drive/MyDrive/KidneyTumorClassification/kidney.zip -d /content/temp_images

In [ ]:
!mv /content/temp_images/kidneyData.csv /content

In [ ]:
!mkdir -p /content/all_images

!find /content/temp_images -name "*.jpg" -exec mv -t /content/all_images {} +

!rm -rf /content/temp_images

In [ ]:
!ls /content/all_images | wc -l

In [ ]:
df = pd.read_csv("kidneyData.csv")

# bináris label: tumor = 1, minden más = 0
df["label"] = df["Class"].apply(lambda x: 1 if x == "Tumor" else 0)

print(df["label"].value_counts())

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# A class_weight-et szigorúan csak a TRAIN adatokból számoljuk ki!
class_counts = train_df["label"].value_counts().sort_index()
total = len(train_df)
weights = total / (2 * class_counts)
class_weights = torch.tensor(weights.values, dtype=torch.float)

print("Osztálysúlyok a tanító adatok alapján:", class_weights)

In [ ]:
normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    normalize
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5

    cm = confusion_matrix(y_true, y_pred)

    report_dict = classification_report(y_true, y_pred, labels=[0, 1], output_dict=True, zero_division=0)
    report_text = classification_report(y_true, y_pred, labels=[0, 1], target_names=['Nem Tumor (0)', 'Tumor (1)'], zero_division=0)

    return {
        "tumor_precision": report_dict['1']['precision'],
        "tumor_recall": report_dict['1']['recall'],
        "tumor_f1": report_dict['1']['f1-score'],

        "non_tumor_precision": report_dict['0']['precision'],
        "non_tumor_recall": report_dict['0']['recall'],
        "non_tumor_f1": report_dict['0']['f1-score'],

        "auc": auc,
        "confusion_matrix": cm,
        "report_text": report_text
    }

In [ ]:
class SkinDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.img_dir, str(row["image_id"]) + ".jpg")
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(row["label"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = SkinDataset(train_df, "/content/all_images/", train_transform)
val_dataset = SkinDataset(val_df, "/content/all_images/", val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
class ResNetModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.resnet18(weights=None)

        num_ftrs = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 1)
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNetModel().to(device)

num_epochs = 50

manual_weight = (class_weights[1] / class_weights[0]) * 0.5
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([manual_weight]).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
#scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

In [ ]:
def count_parameters(model):
    # Végigmegy a modell összes rétegén, és összeadja a tanítható (requires_grad=True) paraméterek számát
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

resnet_model = ResNetModel()
resnet_params = count_parameters(resnet_model)

print(f"ResNet-18 paraméterszáma: {resnet_params:,}")

In [ ]:
start_epoch = 0
best_score = 0
epochs_no_improve = 0

if os.path.exists(last_model_path):
    print("-> Korábbi mentés megtalálva! Folytatás betöltése...")
    checkpoint = torch.load(last_model_path, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_score = checkpoint['best_score']
    epochs_no_improve = checkpoint['epochs_no_improve']

    print(f"-> A tanítás a(z) {start_epoch + 1}. epochától folytatódik.")
else:
    print("-> Nincs korábbi mentés. Tanítás indítása a nulláról.")

In [ ]:
if start_epoch == 0:
    with open(log_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_loss", "val_loss",
            "tumor_precision", "tumor_recall", "tumor_f1",
            "non_tumor_precision", "non_tumor_recall", "non_tumor_f1",
            "auc", "learning_rate"
        ])

In [ ]:
patience = 10

print("Tanítás futtatása...")
for epoch in range(start_epoch, num_epochs):

    # --- TANÍTÁS ---
    model.train()
    train_loss = 0
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)

    for images, labels in train_progress:
        images = images.to(device)
        labels = labels.to(device).float()

        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_progress.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss /= len(train_loader)

    # --- VALIDÁCIÓ ---
    model.eval()
    val_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]", leave=False)

    with torch.no_grad():
        for images, labels in val_progress:
            images = images.to(device)
            labels = labels.to(device).float()

            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

            val_progress.set_postfix({'loss': f"{loss.item():.4f}"})

    val_loss /= len(val_loader)

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    metrics = compute_metrics(all_labels, all_preds, all_probs)

    print(f"\n{'-'*50}")
    print(f"Epoch {epoch+1}/{num_epochs} | LR: {current_lr}")
    print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | AUC: {metrics['auc']:.4f}")
    print("\nClassification Report:\n", metrics["report_text"])
    print("Confusion matrix:\n", metrics["confusion_matrix"])
    print(f"{'-'*50}\n")

    with open(log_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch+1, train_loss, val_loss,
            metrics['tumor_precision'], metrics['tumor_recall'], metrics['tumor_f1'],
            metrics['non_tumor_precision'], metrics['non_tumor_recall'], metrics['non_tumor_f1'],
            metrics['auc'], current_lr
        ])

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_score': best_score,
        'epochs_no_improve': epochs_no_improve
    }

    torch.save(checkpoint, last_model_path)

    score = metrics["auc"]

    if score > best_score:
        best_score = score
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print("-> Új legjobb modell mentve (AUC javult)!")
    else:
        epochs_no_improve += 1
        print(f"-> Nincs javulás {epochs_no_improve} epocha óta.")

    if epochs_no_improve >= patience:
        print(f"Early stopping aktiválva az {epoch+1}. epochánál.")
        break

print("\nTanítás befejezve!")